# THE MOMENT — HunyuanVideo 1.5 I2V on Kaggle 2×T4

Run cells top-to-bottom. First run ~15–20 min (setup + ~15 GB weights).

**If Cell 5 or Cell 7 errors on a node name, paste /kaggle/working/comfyui.log.** The workflow JSON in Cell 4 is a first draft against Comfy-Org's Nov 2025 native HunyuanVideo-1.5 support.


In [ ]:
# === Cell 1: SETUP (mount kit from GODSHAND dataset) ===
import os, shutil
KIT_ROOT = '/kaggle/working'
os.chdir(KIT_ROOT)
kit_src = None
for root, dirs, files in os.walk('/kaggle/input'):
    if os.path.basename(root) == 'src' and 'render_motion.py' in files:
        kit_src = os.path.dirname(root); break
assert kit_src, 'ERROR: no attached dataset contains src/ — attach GODSHAND.'
print(f'[setup] kit at: {kit_src}')
for entry in os.listdir(kit_src):
    s = os.path.join(kit_src, entry); d = os.path.join(KIT_ROOT, entry)
    if os.path.isdir(s):
        if os.path.exists(d): shutil.rmtree(d)
        shutil.copytree(s, d)
    else:
        shutil.copy2(s, d)
for req in ['src/render_motion.py','src/comfy_client.py','workflows/flf2v.json','config.yaml','shots_bangalore.json']:
    assert os.path.exists(req), f'MISSING: {req}'
print('[setup] structure OK')
os.environ['KIT_CONFIG'] = 'config.yaml'
os.environ['PYTHONPATH'] = 'src:' + os.environ.get('PYTHONPATH','')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'
print('[setup] env ready')


In [ ]:
# === Cell 2: INSTALL ComfyUI + ComfyUI-GGUF + VideoHelperSuite ===
import subprocess, sys
from pathlib import Path
COMFY = Path('/kaggle/working/ComfyUI')
def sh(cmd, cwd=None):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout); print(r.stderr, file=sys.stderr)
        raise RuntimeError(f'failed: {cmd}')
if not COMFY.exists():
    sh('git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git', cwd='/kaggle/working')
sh('pip install -q -r requirements.txt', cwd=str(COMFY))
cn = COMFY/'custom_nodes'; cn.mkdir(exist_ok=True)
for name,url in [('ComfyUI-GGUF','https://github.com/city96/ComfyUI-GGUF.git'),
                 ('ComfyUI-VideoHelperSuite','https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git')]:
    dst = cn/name
    if not dst.exists(): sh(f'git clone --depth 1 {url}', cwd=str(cn))
sh('pip install -q gguf sentencepiece protobuf einops')
print('[install] ready')


In [ ]:
# === Cell 3: DOWNLOAD HunyuanVideo 1.5 I2V models ===
from pathlib import Path
import shutil
from huggingface_hub import hf_hub_download, list_repo_files
COMFY = Path('/kaggle/working/ComfyUI'); MODELS = COMFY/'models'
for sub in ['unet','vae','text_encoders','clip_vision']:
    (MODELS/sub).mkdir(parents=True, exist_ok=True)
GGUF_REPO = 'jayn7/HunyuanVideo-1.5_I2V_480p-GGUF'
gguf_files = [f for f in list_repo_files(GGUF_REPO) if f.startswith('480p_distilled/') and f.endswith('.gguf')]
pref = ['Q4_K_S','Q4_K_M','Q4_K','Q4_0','Q6_K','Q8_0']
picked = next((f for q in pref for f in gguf_files if q.lower() in f.lower()), gguf_files[0] if gguf_files else None)
assert picked, 'no GGUF found'
local = hf_hub_download(GGUF_REPO, filename=picked, local_dir=str(MODELS/'unet'), local_dir_use_symlinks=False)
flat = MODELS/'unet'/Path(picked).name
if not flat.exists(): shutil.move(local, flat)
GGUF_NAME = flat.name
CO_REPO = 'Comfy-Org/HunyuanVideo_1.5_repackaged'
for path, dest in [('split_files/vae/hunyuanvideo15_vae_fp16.safetensors', MODELS/'vae'),
                   ('split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors', MODELS/'text_encoders'),
                   ('split_files/text_encoders/byt5_small_glyphxl_fp16.safetensors', MODELS/'text_encoders')]:
    local = hf_hub_download(CO_REPO, filename=path, local_dir=str(COMFY), local_dir_use_symlinks=False)
    flat = dest/Path(path).name
    if not flat.exists(): shutil.move(local, flat)
open('/kaggle/working/.gguf_name','w').write(GGUF_NAME)
print(f'[download] done. GGUF: {GGUF_NAME}')


In [ ]:
# === Cell 4: OVERWRITE kit workflows with HunyuanVideo 1.5 I2V graph ===
import json
from pathlib import Path
GGUF_NAME = open('/kaggle/working/.gguf_name').read().strip()
WORKFLOW = '{\n  "1": {\n    "class_type": "UnetLoaderGGUF",\n    "inputs": {\n      "unet_name": "__GGUF_FILENAME__"\n    },\n    "_meta": {\n      "title": "HunyuanVideo 1.5 I2V (GGUF)"\n    }\n  },\n  "2": {\n    "class_type": "DualCLIPLoader",\n    "inputs": {\n      "clip_name1": "qwen_2.5_vl_7b_fp8_scaled.safetensors",\n      "clip_name2": "byt5_small_glyphxl_fp16.safetensors",\n      "type": "hunyuan_video"\n    },\n    "_meta": {\n      "title": "Text encoders"\n    }\n  },\n  "3": {\n    "class_type": "VAELoader",\n    "inputs": {\n      "vae_name": "hunyuanvideo15_vae_fp16.safetensors"\n    },\n    "_meta": {\n      "title": "HunyuanVideo 1.5 VAE"\n    }\n  },\n  "4": {\n    "class_type": "CLIPTextEncode",\n    "inputs": {\n      "text": "POSITIVE",\n      "clip": [\n        "2",\n        0\n      ]\n    },\n    "_meta": {\n      "title": "Positive"\n    }\n  },\n  "5": {\n    "class_type": "CLIPTextEncode",\n    "inputs": {\n      "text": "NEGATIVE",\n      "clip": [\n        "2",\n        0\n      ]\n    },\n    "_meta": {\n      "title": "Negative"\n    }\n  },\n  "6": {\n    "class_type": "LoadImage",\n    "inputs": {\n      "image": "start.png"\n    },\n    "_meta": {\n      "title": "Input still"\n    }\n  },\n  "7": {\n    "class_type": "LoadImage",\n    "inputs": {\n      "image": "start.png"\n    },\n    "_meta": {\n      "title": "End frame stub (kit compat)"\n    }\n  },\n  "14": {\n    "class_type": "HunyuanVideo15ImageToVideo",\n    "inputs": {\n      "positive": [\n        "4",\n        0\n      ],\n      "negative": [\n        "5",\n        0\n      ],\n      "vae": [\n        "3",\n        0\n      ],\n      "start_image": [\n        "6",\n        0\n      ],\n      "width": 720,\n      "height": 480,\n      "length": 121,\n      "batch_size": 1\n    },\n    "_meta": {\n      "title": "HunyuanVideo 1.5 I2V"\n    }\n  },\n  "15": {\n    "class_type": "KSampler",\n    "inputs": {\n      "model": [\n        "1",\n        0\n      ],\n      "positive": [\n        "14",\n        0\n      ],\n      "negative": [\n        "14",\n        1\n      ],\n      "latent_image": [\n        "14",\n        2\n      ],\n      "steps": 8,\n      "cfg": 7.0,\n      "sampler_name": "uni_pc",\n      "scheduler": "beta",\n      "seed": 42,\n      "denoise": 1.0\n    },\n    "_meta": {\n      "title": "Sample (8-step distilled)"\n    }\n  },\n  "16": {\n    "class_type": "VAEDecode",\n    "inputs": {\n      "samples": [\n        "15",\n        0\n      ],\n      "vae": [\n        "3",\n        0\n      ]\n    },\n    "_meta": {\n      "title": "Decode"\n    }\n  },\n  "17": {\n    "class_type": "VHS_VideoCombine",\n    "inputs": {\n      "images": [\n        "16",\n        0\n      ],\n      "frame_rate": 24,\n      "loop_count": 0,\n      "filename_prefix": "hunyuan_i2v_take",\n      "format": "video/h264-mp4",\n      "pix_fmt": "yuv420p",\n      "crf": 18,\n      "save_output": true\n    },\n    "_meta": {\n      "title": "Combine -> mp4"\n    }\n  },\n  "_bind": {\n    "positive": [\n      "4",\n      "text"\n    ],\n    "negative": [\n      "5",\n      "text"\n    ],\n    "width": [\n      "14",\n      "width"\n    ],\n    "height": [\n      "14",\n      "height"\n    ],\n    "seconds": [\n      "14",\n      "length"\n    ],\n    "steps": [\n      "15",\n      "steps"\n    ],\n    "cfg": [\n      "15",\n      "cfg"\n    ],\n    "seed": [\n      "15",\n      "seed"\n    ],\n    "ref_dirs": [\n      "6",\n      "image"\n    ]\n  }\n}'
wf = json.loads(WORKFLOW)
wf['1']['inputs']['unet_name'] = GGUF_NAME
for name in ['flf2v','phantom','s2v']:
    (Path('/kaggle/working/workflows')/f'{name}.json').write_text(json.dumps(wf, indent=2))
print('[workflows] done — every shot routes through HunyuanVideo 1.5 I2V')


In [ ]:
# === Cell 4a: PATCH config.yaml resolution for 480p model on T4 ===
import re
cfg_path = '/kaggle/working/config.yaml'
src = open(cfg_path).read()
src = re.sub(r'^(\s*width:\s*)\d+',  r'\g<1>720', src, count=1, flags=re.M)
src = re.sub(r'^(\s*height:\s*)\d+', r'\g<1>480', src, count=1, flags=re.M)
open(cfg_path,'w').write(src)
print('[config] width: 720, height: 480 (matches 480p HunyuanVideo)')


In [ ]:
# === Cell 5: LAUNCH ComfyUI + health check ===
import subprocess, time, urllib.request, sys
from pathlib import Path
COMFY = Path('/kaggle/working/ComfyUI'); LOG = Path('/kaggle/working/comfyui.log')
subprocess.run(['pkill','-f','ComfyUI/main.py'], capture_output=True); time.sleep(2)
proc = subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--cache-none','--disable-smart-memory'],
                       cwd=str(COMFY), stdout=open(LOG,'w'), stderr=subprocess.STDOUT)
print(f'[comfyui] PID {proc.pid}; log -> {LOG}')
deadline = time.time() + 300; last_err = None
while time.time() < deadline:
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=5) as r:
            print('[comfyui] ready'); break
    except Exception as e:
        last_err = e; time.sleep(3)
else:
    print(open(LOG).read()[-3000:]); raise RuntimeError(f'ComfyUI never bound: {last_err}')


In [ ]:
# === Cell 6: SMOKE TEST ===
import os
os.environ.pop('COMFY_MOCK', None)
!KIT_CONFIG=config.yaml python3 src/selftest.py


In [ ]:
# === Cell 7: COST METER (1 real sample shot) ===
!KIT_CONFIG=config.yaml python3 src/cost_meter.py --sample-shot p01_thin_boy


In [ ]:
# === Cell 8: FULL RUN — all 39 shots ===
import os
os.environ['CONFIRM_FULL_RUN'] = '1'
os.environ['KIT_RENDER_MODE']  = 'motion_stills'
os.environ['MAX_HOURS']        = '8'
!KIT_CONFIG=config.yaml python3 run.py
